COMPONENTES:
    
    aluno: Mateus Andrade Antezana
    RA: 24021000

### Importacoes

In [ ]:
import pandas as pd
import seaborn as sns
import plotly.express as px
import numpy as np
import matplotlib.pyplot as plt
from random import uniform, randint, sample

### Defs

In [ ]:
def valor_(a:float|int, b:float|int)->float|int:
    """
    RESUMO
        Recebe dois parametros, faz sua subtracao e potencia.
    PARAMETROS
        a: float ou int
        b: float ou int
    RETORNA
        float ou int
    """
    try:
        a = float(a)
        b = float(b)
        valor:float = (a-b)**2
        return valor
    except:
        raise ValueError(f"Valores {a} ou {b} nao sao numericos")

def distancia_euclidiana(linha_um:list[float], linha_dois:list[float])->float:
    """
    RESUMO
        Calcula a distancia entre as linhas index_um e index_dois.
        Quanto mais proximo de 0, mais proximos, ou relacionados,
            sao index_um e index_dois.
    PARAMETROS
        df -> seu dataframe
        index_um -> um index qualquer do seu dataframe
        index_dois -> um outro index qualquer do seu dataframe
    RETORNO
        float
    """
    distancia:float = 0
    for index in range(len(linha_um)):
        l1:list[float]|float = linha_um[index] if(isinstance(linha_um, list)) else linha_um
        l2:list[float]|float = linha_dois[index] if(isinstance(linha_dois, list)) else linha_dois
        # print(f"{l1}, {l2}")
        distancia += valor_(l1, l2)
    return float(np.sqrt(distancia))

def gerar_clusters(df:pd.DataFrame, qntd:int)->list:
    """
    RESUMO
        A partir de um dataframe, conta a quantidade de linhas
        e retorna linhas clusters centrais aleatorias para serem
        usadas como base de grupo.
    PARAMETROS
        df -> um dataframe
        qntd -> um numero inteiro
    RETORNA
        list[list[float],...]
    """
    tmnh_df = len(df)
    valores:list[int] = []
    for i in range(qntd):
        n:int = randint(0, tmnh_df-1)
        while(n in valores):
            n:int = randint(0, tmnh_df-1)
        valores.append((df.iloc[n].values).tolist())
    return valores

def matrix_temporaria(df:pd.DataFrame, centroides_provisorios:list[int])->list[float]:
    """
    RESUMO
        Forma matrix temporaria a partir dos clusters iniciais.
    PARAMETROS
        df -> o dataframe selecionado
        centroides_provisorios -> lista int de indices para localizar no df
    RETORNO
        list[float]
    """
    matrix:list[float] = []
    for index, cluster in enumerate(centroides_provisorios):
        lista:list[float] = []
        for i in range(len(df)):
            centroide:list[float]|float = centroides_provisorios[index]
            if(isinstance(centroide, list)):
                lista.append(distancia_euclidiana((df.iloc[i].values).tolist(), centroides_provisorios[index]))
            else:
                lista.append(distancia_euclidiana((df.iloc[i].values).tolist(), cluster))
        matrix.append(lista)
    return matrix

def matrix_de_pertinencia(matrix:list[float], cluster:int, m:int)->list[float]:
    """
    RESUMO
        Define a matrix de pertinencia. Mostra o grau de semelhanca de cada
        linha referente a coluna do cluster.
    PARAMETROS
        matrix -> uma lista de valores flutuantes
        cluster -> numero inteiro
        m -> numero inteiro
    RETORNA
        list[list[float],...]
    """
    matrix_U:list[float] = []
    for i in range(len(matrix[0])):
        lista_cluster:list[float] = []
        for c in range(0, cluster):
            lista_cluster.append(matrix[c][i])
            # print(lista_cluster)
        lista_U:list[float] = []
        for j, posj in enumerate(lista_cluster):
            den_U:float = 0
            for k, posk in enumerate(lista_cluster):
                if(posk==1):
                    den_U+=1
                elif(posk==0):
                    den_U+=0
                else:
                    den_U += ((posj/posk)**(2/(m-1)))
            if(den_U>0):
                lista_U.append(1/den_U)
            else:
                lista_U.append(0.0)
        # print(lista_U)
        matrix_U.append(lista_U)
    return matrix_U

def encontrar_centros(df:pd.DataFrame, matrix_U:list[list[float]], qntd_clusters:int, m:int)->list[float]:
    """
    RESUMO
        Encontra os possiveis centroides.
    PARAMETROS
        matrix_U -> espera uma lista de floats. A matrix de Pertinencia
        qntd_clusters -> a quantidade definida de grupos
        m -> numero inteiro que foi selecionado
    RETORNO
        list[float]
    """
    centros = []
    for cluster in range(qntd_clusters):
        centro_c:list[float] = []
        for c in df.columns:
            soma_col:float = 0.0
            denominador = 0.0
            for index in range(len(df)):
                u = matrix_U[index][cluster] ** m
                soma_col += df[c].iloc[index] * u
                denominador += u
                # print(f"u:{u} # Somatorio:{soma_col} # Denominador:{denominador}")
            centro_c.append(float(soma_col / denominador))
        centros.append(centro_c)
    return centros

def fuzzy_cmeans(df:pd.DataFrame, grupos:int, m:int)->list[list[float]]:
    """
    RESUMO
        Cria grupos e "suaviza" a quais pertence, podendo se
            encaixar em mais de um grupo.
    PARAMETROS
        df -> aceita tanto um dataset quanto um dataframe
        grupos -> quantidade de grupos
        m -> numero inteiro
    RETORNO
        list[list[float]]
    """
    clusters:list[list[float]] = gerar_clusters(df, grupos)
    copia:list[list[float]] = clusters
    i:int = 0
    for i in range(100):
        matrix:list[float] = matrix_temporaria(df, clusters)
        matrix_U = matrix_de_pertinencia(matrix, len(clusters), m)
        clusters = encontrar_centros(df, matrix_U, len(clusters), m)
        if(any(clusters[i]==copia[i] for i in range(len(clusters)))):
            # print("Sao iguais")
            break
        else:
            # print("Nao sao iguais")
            copia = clusters
        # print(clusters)
    # print(f"Foram necessarios {i} iteracoes.")
    return clusters, matrix_U

### Principal

#### Carregando dataset e tornando dataset em numerico

In [ ]:
iris = sns.load_dataset('iris')
iris = iris.select_dtypes(include=['float64'])

#### Rodando fuzzy cmeans e mostrando Matrix de Pertinência

Questão 2:

Aplicar o algoritmo na base de dados Iris (completa, 4 atributos, 3 classes) e listar as pertinências de cada amostra usando 3 grupos

In [ ]:
m:int = 2
grupos:int = 3
clusters, matrix_U = fuzzy_cmeans(iris, grupos, m)
colunas:list[str] = [f"Grupo {i}" for i in range(len(clusters))]
matrix_U = pd.DataFrame(data=matrix_U, columns=colunas)

In [ ]:
matrix_U

,Grupo 0,Grupo 1,Grupo 2
0,0.002304,0.996624,0.001072
1,0.016650,0.975853,0.007498
2,0.013759,0.979826,0.006415
3,0.022465,0.967427,0.010108
4,0.003762,0.994470,0.001768
...,...,...,...
145,0.106387,0.011262,0.882351
146,0.507525,0.025796,0.466679
147,0.156440,0.012114,0.831447
148,0.189036,0.021581,0.789382


#### Plotando gráfico com a suavidade do fuzzy c-means

Questão 3:

Plot de resultados utilizando a base de dados Iris (apenas as amostras     virgínica e versicolor) usando 2 grupos. Notem que é importante demonstrar a pertinência intermediária, o que será mais fácil em um problema de binário

In [ ]:
iris_2:pd.DataFrame = sns.load_dataset('iris')

iris_binario = iris_2[iris_2['species'].isin(['virginica', 'versicolor'])].copy()
iris_binario_num = iris_binario.select_dtypes(include=['float64'])

m:int = 2
grupos:int = 2
centros_binario, matrix_U_binario = fuzzy_cmeans(iris_binario_num, grupos, m)

pertinencia_grupo_0 = [linha[0] for linha in matrix_U_binario]
iris_binario['Grau_Pertinencia'] = pertinencia_grupo_0

px.scatter(
    iris_binario,
    x='petal_length',
    y='petal_width',
    color='Grau_Pertinencia',
    color_continuous_scale='RdBu',
    title='Fuzzy C-Means Binário: Virginica (Azul) vs Versicolor (Vermelho)'
).show()